# FTS Custom ML Backtesting Workspace

Welcome to the custom backtesting workspace! This notebook demonstrates how to run a realistic simulation of a machine learning-based trading strategy using the Financial Trading System (FTS) framework.

### Features of this Backtest:
1. **Historical Replay Event Loop:** Ticks are played back chronologically from our SQLite database.
2. **LSTM Forecasting Strategy:** Uses a pre-trained LSTM neural network to predict price direction from historical close prices.
3. **Execution Delay:** Simulates a realistic **1-bar execution delay** using `KBarExecuteDelay(k=1)` (meaning a signal generated at tick $T$ is submitted, and execution/fill status is checked at tick $T+1$).
4. **Price Slippage:** Simulates **0.1% price slippage** on both buy and sell orders using `FlatPriceSlip`.
5. **SOLID Design:** Directly uses the standard FTS pipeline components (no custom subclasses or leaky visualizer hacks needed).
6. **Visual Inspection:** Uses the `BacktestVisualizer` to display predictions overlaid with buy/sell trade markers (derived cleanly from the filled `order_logs`).
7. **Performance Summary Metrics:** Calculates and exports annualized returns, Sharpe ratio, and maximum drawdown metrics.

### 1. Import Dependencies

In [1]:
import os
import json
import logging
import pandas as pd
import numpy as np
from datetime import datetime, timezone
from sqlalchemy import create_engine

# Core FTS components
from trading_bot.config import settings
from trading_bot.core.database import init_db, SessionLocal
from trading_bot.core.loop import HistoricalReplayLoop
from trading_bot.core.pipeline import TradingPipeline
from trading_bot.monitoring.prediction_logger import DatabasePredictionLogger
from trading_bot.core.models import BacktestPredictionLog, OrderLog as OrderLogModel, TradeLog as TradeLogModel, Position as PositionModel
from trading_bot.core.repository import MarketDataRepository, OrderRepository, PositionRepository
from trading_bot.core.schemas import BarData, OrderSide, OrderStatus

# ML/Strategy and Risk components
from nets.output_selectors import DynamicThresholdClassifier
from nets.inference import ONNXPredictor
from nets.strategies.nets_strategy import NetsStrategy
from trading_bot.core.transforms import LogReturnTransform
from trading_bot.strategy.engine import StrategyEngine
from trading_bot.risk_management.portfolio import Portfolio
from trading_bot.risk_management.sizing.fixed_percentage import FixedPercentageSizer
from trading_bot.risk_management.manager import RiskManager

# Execution & Backtest components
from trading_bot.execution.delay import KBarExecuteDelay
from trading_bot.execution.slippage import FlatPriceSlip
from trading_bot.execution.handlers.simulated_handler import SimulatedExecutionHandler
from trading_bot.execution.engine import ExecutionEngine
from trading_bot.backtesting.readers import SQLBacktestDataReader
from trading_bot.backtesting import BacktestVisualizer, HTMLBacktestExporter

# Set logging level to INFO for detailed simulation traces
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()

### 2. Setup SQLite Database & Connect

We connect to the local SQLite database and clear any existing logs associated with our specific `run_id` to ensure a clean backtest run, without dropping the tables.

In [2]:
db_url = 'sqlite:///../dev.db'
settings.DATABASE_URL = "sqlite+pysqlite:///../dev.db"

engine = create_engine(db_url, pool_pre_ping=True)
SessionLocal.configure(bind=engine)
db = SessionLocal()

run_id = 'backtest_custom_lstm'

# Clear logs from previous runs of this specific backtest to ensure clean metrics
db.query(BacktestPredictionLog).filter_by(run_id=run_id).delete()
db.query(OrderLogModel).filter_by(run_id=run_id).delete()
db.query(TradeLogModel).filter_by(run_id=run_id).delete()
db.query(PositionModel).filter_by(run_id=run_id).delete()
db.commit()

print("Connected to database and cleared logs for run ID:", run_id)

Connected to database and cleared logs for run ID: backtest_custom_lstm


### 3. Setup Strategy and Prediction Logic

We load our pre-trained LSTM model from `models/my_lstm_model.onnx`, set up feature transformation using logarithmic returns, and configure a threshold classifier output selector.

In [3]:
onnx_path = '../models/my_lstm_model.onnx'                               
if not os.path.exists(onnx_path):                                        
    onnx_path = 'models/my_lstm_model.onnx'

print("Loading ONNX predictor from:", onnx_path)
predictor = ONNXPredictor(onnx_path)
transform = LogReturnTransform()
output_selector = DynamicThresholdClassifier(k=0.03, period=10,          
confidence_multiplier=20.0)

strategy = NetsStrategy(
    predictor=predictor,
    transform=transform,
    output_selector=output_selector,
    lookback_period=20,
    name_suffix='lstm',
    feature_cols=['close']
)
strategy_engine = StrategyEngine(strategies=[strategy])


2026-06-25 22:46:52,007 - WARNING - Could not parse ONNX model metadata for ../models/my_lstm_model.onnx: Missing required ONNX model metadata key: 'train_start_date'
2026-06-25 22:46:52,008 - INFO - Loaded ONNX model from ../models/my_lstm_model.onnx
2026-06-25 22:46:52,009 - INFO - StrategyEngine initialized with 1 strategies: [nets_strategy_lstm]


Loading ONNX predictor from: ../models/my_lstm_model.onnx


### 4. Build Backtesting Pipeline with Delay and Slippage Models

Here we instantiate the components required for a realistic backtest simulation:
- **Execution Delay:** `KBarExecuteDelay(k=1)` is injected into the simulated handler.
- **Slippage:** `FlatPriceSlip(slippage_pct=0.001)` (0.1% price penalty) is injected into the simulated handler.
- **Portfolio & Sizer:** A standard portfolio initialized with $10,000, sizing positions at 10% of total equity per trade.

In [4]:
pos_repo = PositionRepository(db)
order_repo = OrderRepository(db)

portfolio = Portfolio(
    initial_balance=10000.0,
    quote_currency="USD",
    pos_repo=pos_repo,
    order_repo=order_repo
)
portfolio._positions = {}

sizer = FixedPercentageSizer(default_percentage=0.10)
risk_manager = RiskManager(portfolio=portfolio, sizer=sizer)

# Define delayed execution (1 bar delay) and slippage (0.1% penalty)
delay_model = KBarExecuteDelay(k=1)
slippage_model = FlatPriceSlip(slippage_pct=0.001)

execution_handler = SimulatedExecutionHandler(
    delay_model=delay_model,
    slippage_model=slippage_model,
    execution_price_source="close",
    initial_balances={"USD": 10000.0}
)

execution_engine = ExecutionEngine(
    execution_handler=execution_handler,
    portfolio=portfolio,
    run_id=run_id
)

pipeline = TradingPipeline(
    ingestion=None,
    strategy=strategy_engine,
    risk=risk_manager,
    execution=execution_engine,
    portfolio=portfolio
)

prediction_logger = DatabasePredictionLogger(
    db=db,
    commit=False,
    model_class=BacktestPredictionLog,
    run_id=run_id
)
pipeline.prediction_logger = prediction_logger

2026-06-25 22:46:52,052 - INFO - Portfolio initialized with cash: 10000.00 USD
2026-06-25 22:46:52,056 - INFO - FixedPercentageSizer initialized with percentage: 10.00%
2026-06-25 22:46:52,058 - INFO - RiskManager initialized with sizer: fixed_percentage, max_allocation: 25.0%, max_positions: 10
2026-06-25 22:46:52,060 - INFO - ExecutionEngine initialized with handler for: simulated, max_retries=3, auto_reconciliation=True


### 5. Run Replay Loop & Record Portfolio Equity

We initialize the data reader to stream BTC/USDT bars between June 1st, 2026, and June 21st, 2026. During the execution of the replay loop, we record the portfolio's cash, position value, and total equity at each tick to construct our equity curve.

In [ ]:
start_date = datetime(2026, 6, 1, 3, 30, 0, tzinfo=timezone.utc)
end_date = datetime(2026, 6, 21, 23, 0, 0, tzinfo=timezone.utc)

data_reader = SQLBacktestDataReader(
    session=db,
    market_id='BTC/USDT',
    start_date=start_date,
    end_date=end_date,
    warmup_bars=100,
    lookback_limit=1000
)
loop_driver = HistoricalReplayLoop(data_reader=data_reader)

print("Starting simulation loop...")
from trading_bot.backtesting.engine import BacktestEngine

# 5. Initialize and Run Backtest Engine
backtest_engine = BacktestEngine(
    pipeline=pipeline,
    data_reader=data_reader,
    db=db,
    market_id='BTC/USDT'
)

# Run simulation and clear previous DB entries matching run_id
result = backtest_engine.run(run_id=run_id, clear_previous_run=True)

# 6. Extract and Save Performance Summary Stats
summary = result.to_dict()
result.save_summary(f"../runs/reports/backtest_summary_{run_id}.json")

print("--- BACKTEST SUMMARY STATS ---")
print(json.dumps(summary, indent=4))

2026-06-25 22:46:52,190 - INFO - Clearing previous database logs for run_id: backtest_custom_lstm


Starting simulation loop...


2026-06-25 22:46:52,208 - INFO - Starting backtest engine simulation for run_id: backtest_custom_lstm
2026-06-25 22:46:52,209 - INFO - Starting HistoricalReplayLoop simulation...
2026-06-25 22:46:52,232 - INFO - StrategyEngine generated a total of 1 signals this tick.
2026-06-25 22:46:52,237 - INFO - RiskManager approved order for BTC/USDT: buy 0.01 shares @ $71711.8900
2026-06-25 22:46:52,238 - INFO - Risk approved order for BTC/USDT (size: 0.0139 shares).
2026-06-25 22:46:52,239 - INFO - [nets_strategy_lstm] Executing order for BTC/USDT: buy 0.0139 shares @ $71711.8900
2026-06-25 22:46:52,239 - INFO - [SimulatedExecutionHandler] Queued order sim-6387a969 (buy) at tick 21, scheduled to fill at tick 22.
2026-06-25 22:46:52,240 - INFO - Handler returned result for BTC/USDT: ID: sim-6387a969, Status: OrderStatus.OPEN
2026-06-25 22:46:52,259 - INFO - [SimulatedExecutionHandler] Filled order sim-6387a969 at price 71603.5320 (base price: 71532.0000, side: buy).
2026-06-25 22:46:52,261 - INF

--- BACKTEST SUMMARY STATS ---
{
    "run_id": "backtest_custom_lstm",
    "market_id": "BTC/USDT",
    "strategy_name": "nets_strategy_lstm",
    "initial_equity": 10000.0,
    "final_equity": 7024.588036059387,
    "total_return_pct": -29.754119639406124,
    "max_drawdown_pct": -33.825638336699235,
    "sharpe_ratio": -6.5675239383279775,
    "total_trades": 745
}


### 7. Render Interactive Dashboard

We load our interactive `BacktestVisualizer` pointing to the SQLite database and render the interactive dashboard to visually inspect cumulative returns, positions, and trades overlaying the candlestick chart.

In [6]:
# Instantiate the visualizer pointing to the database
viz = BacktestVisualizer('sqlite:///../dev.db')

# Display the dashboard (incorporates LSTM ONNX model structure details if available)
viz.show_dashboard(onnx_model_path=onnx_path)

Output()

Output()

HTML(value="<hr style='border-color:#37474f;'/>")

HTML(value='<h3>🧬 ONNX Model Weight Inspector</h3>')

Output()

### 8. Export Standalone Interactive Visualization Report

Finally, we write the entire interactive visualization charts out to a standalone HTML file inside `runs/reports/` for offline review.

In [7]:
exporter = HTMLBacktestExporter(visualizer=viz)
report_path = exporter.export(
    market_id='BTC/USDT',
    strategy_name=strategy.name,
    run_id=run_id,
    output_path='../runs/reports/'
)
print("Interactive HTML report successfully exported to:", report_path)
db.close()

2026-06-25 22:47:48,383 - INFO - Saving interactive backtest visualization HTML to: ../runs/reports/report_BTC_USDT_nets_strategy_lstm_backtest_custom_lstm.html


Interactive HTML report successfully exported to: ../runs/reports/report_BTC_USDT_nets_strategy_lstm_backtest_custom_lstm.html
